# 🎙️ Préparation du Dataset Reverb-TTS (Français — 24 kHz + Verbalis)

Ce notebook prépare un dataset propre pour **Reverb-TTS** (`software-mansion/Reverb-TTS`) sur Google Colab :
1. **Découpe sur les vrais silences** (aucun mot coupé au début ou à la fin).
2. **Format audio natif Reverb / Higgs Audio v2** : **24 000 Hz, mono, PCM 16-bit**, normalisation des passages trop faibles.
3. **Transcription Whisper `large-v3`** clip par clip.
4. **Normalisation texte obligatoire `Verbalis`** (nombres, dates, symboles et abréviations en toutes lettres en français).
5. **Filtrage locuteur + qualité**, split 90/10 et export vers **`Mon Drive/reverb_tts/dataset_reverb_narrateur.zip`**.

### Avant de lancer
1. Placez vos fichiers audio (WAV/MP3 ou ZIP) dans **`Mon Drive/thierry_blanc/`**.
2. Dans Colab : **Exécution → Modifier le type d'exécution → GPU (T4, L4 ou A100)**.

In [ ]:
# ===== CONFIGURATION =====
SIM_THRESHOLD = 0.75   # similarité vocale minimale avec le narrateur
MIN_LOGPROB = -0.60    # score minimal de confiance Whisper
SPLIT_SEED = 42
DEV_RATIO = 0.10

# Découpe sur silences (Reverb fonctionne idéalement entre 3 s et 12-15 s)
TARGET_SR = 24000      # 24 kHz obligatoire pour le codec Higgs Audio v2 de Reverb-TTS
MIN_CLIP = 3.0
MAX_CLIP = 14.5
SILENCE_DB = -38.0
MIN_SILENCE_S = 0.25
GAP_MAX = 1.2

DRIVE_INPUT = "/content/drive/MyDrive/thierry_blanc"
DRIVE_OUTPUT = "/content/drive/MyDrive/reverb_tts"
NARRATOR_CLIP_ID = ""  # Laissez vide au 1er passage, puis renseignez l'ID du clip narrateur à l'étape 5

## 1. Connexion Google Drive & Installation des dépendances

In [ ]:
import os
from google.colab import drive

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
!pip install -q faster-whisper soundfile torchaudio numpy scipy resemblyzer verbalis nvidia-cublas-cu12 nvidia-cudnn-cu12

## 2. Repérage des fichiers audio sources

In [ ]:
import glob, zipfile

WORK = "/content/work_reverb_prep"
os.makedirs(WORK, exist_ok=True)

VALID_EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg")
FILES = []
for root, _, fns in os.walk(DRIVE_INPUT):
    for fn in sorted(fns):
        p = os.path.join(root, fn)
        if fn.lower().endswith(".zip"):
            unz = os.path.join(WORK, "unzipped_src")
            os.makedirs(unz, exist_ok=True)
            with zipfile.ZipFile(p, "r") as z:
                z.extractall(unz)
            for r2, _, fns2 in os.walk(unz):
                for fn2 in sorted(fns2):
                    if fn2.lower().endswith(VALID_EXTS):
                        FILES.append(os.path.join(r2, fn2))
        elif fn.lower().endswith(VALID_EXTS):
            FILES.append(p)

FILES = sorted(set(FILES))
assert FILES, f"Aucun fichier audio trouvé dans {DRIVE_INPUT}"
print(f"📁 {len(FILES)} fichier(s) audio trouvé(s).")

## 3. Découpe sur silences (24 kHz) + Transcription Whisper + Normalisation Verbalis

In [ ]:
import ctypes, re, unicodedata
import numpy as np
import soundfile as sf
import torch, torchaudio
import IPython.display as ipd
from faster_whisper import WhisperModel
from verbalis import Verbalizer

if torch.cuda.is_available():
    import nvidia
    libs = []
    for base in getattr(nvidia, "__path__", []):
        libs.extend(glob.glob(f"{base}/*/lib/libcublas*.so*"))
        libs.extend(glob.glob(f"{base}/*/lib/libcudnn*.so*"))
    for _lib in sorted(libs):
        ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)

WHISPER_SR = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
model = WhisperModel("large-v3", device=DEVICE, compute_type=COMPUTE)
verbalizer = Verbalizer("fr")
BRACKETS = re.compile(r"[()\[\]{}]")

def prepare_reverb_text(text):
    # Normalisation stricte identique à reverb_tts/text.py
    t = verbalizer.process(text)
    return " ".join(BRACKETS.sub("", t).split())

def load_audio_24k(path):
    try:
        data, sr = sf.read(path, dtype="float32")
    except Exception:
        import subprocess
        tmp = "/tmp/_decode_24k.wav"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(TARGET_SR), tmp], check=True)
        data, sr = sf.read(tmp, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != TARGET_SR:
        data = torchaudio.functional.resample(torch.from_numpy(data), sr, TARGET_SR).numpy()
    return data.astype(np.float32), TARGET_SR

def split_audio_by_silences(audio, sr):
    hop = int(0.01 * sr)
    win = int(0.02 * sr)
    n_frames = max(1, (len(audio) - win) // hop + 1)
    sq = audio * audio
    csum = np.cumsum(np.insert(sq, 0, 0.0))
    starts = np.arange(n_frames) * hop
    rms = np.sqrt(np.maximum(1e-12, (csum[starts + win] - csum[starts]) / win))
    ref_rms = np.percentile(rms, 95)
    thresh = ref_rms * (10.0 ** (SILENCE_DB / 20.0))
    is_speech = rms > thresh

    min_sil_frames = int(MIN_SILENCE_S / 0.01)
    speech_segs, in_sp, sp_start, sil_count = [], False, 0, 0
    for idx, sp in enumerate(is_speech):
        if sp:
            if not in_sp:
                sp_start, in_sp = idx, True
            sil_count = 0
        elif in_sp:
            sil_count += 1
            if sil_count >= min_sil_frames:
                sp_end = idx - sil_count + 1
                speech_segs.append([sp_start * hop, min(len(audio), sp_end * hop + win)])
                in_sp, sil_count = False, 0
    if in_sp:
        speech_segs.append([sp_start * hop, len(audio)])

    def refine(s_idx, e_idx, depth=0):
        if (e_idx - s_idx) / sr <= MAX_CLIP:
            return [(s_idx, e_idx)]
        if depth > 4:
            return []
        margin = int(0.25 * (e_idx - s_idx))
        f_a, f_b = (s_idx + margin) // hop, min(len(rms), (e_idx - margin) // hop)
        sub = rms[f_a:f_b]
        if len(sub) <= 12:
            return []
        smooth = np.convolve(sub, np.ones(12) / 12, mode="valid")
        best_f = f_a + int(np.argmin(smooth)) + 6
        if rms[best_f] > thresh * 2.5:
            return []
        cut = best_f * hop
        return refine(s_idx, cut, depth + 1) + refine(cut, e_idx, depth + 1)

    refined = []
    for s_idx, e_idx in speech_segs:
        refined.extend(refine(s_idx, e_idx))

    pad = int(0.10 * sr)  # 100 ms de silence propre (norme Reverb trim_silence)
    merged, cur_s, cur_e = [], None, None
    for s_idx, e_idx in refined:
        if cur_s is None:
            cur_s, cur_e = s_idx, e_idx
        elif (s_idx - cur_e) / sr <= GAP_MAX and (e_idx - cur_s) / sr <= MAX_CLIP:
            cur_e = e_idx
        else:
            if (cur_e - cur_s) / sr >= MIN_CLIP:
                merged.append((max(0, cur_s - pad), min(len(audio), cur_e + pad)))
            cur_s, cur_e = s_idx, e_idx
    if cur_s is not None and (cur_e - cur_s) / sr >= MIN_CLIP:
        merged.append((max(0, cur_s - pad), min(len(audio), cur_e + pad)))
    return merged

AUDIO_OUT_DIR = f"{WORK}/data/audio"
os.makedirs(AUDIO_OUT_DIR, exist_ok=True)
rows, durs = {}, {}
fade_len = int(0.01 * TARGET_SR)

for f in FILES:
    audio, sr = load_audio_24k(f)
    bounds = split_audio_by_silences(audio, sr)
    print(f"{os.path.basename(f)} -> {len(bounds)} segments")
    for s_idx, e_idx in bounds:
        clip = audio[s_idx:e_idx].copy()
        # Normalisation si trop faible (< -40 dBFS RMS) comme dans reverb_tts/audio.py
        clip_rms = float(np.sqrt(np.mean(clip ** 2))) if len(clip) else 0.0
        if clip_rms > 0 and 20 * np.log10(clip_rms) < -40.0:
            peak = float(np.max(np.abs(clip)))
            clip *= min((10 ** (-20.0 / 20)) / clip_rms, 0.99 / max(peak, 1e-8))
        if len(clip) > 2 * fade_len:
            clip[:fade_len] *= np.linspace(0.0, 1.0, fade_len, dtype=np.float32)
            clip[-fade_len:] *= np.linspace(1.0, 0.0, fade_len, dtype=np.float32)

        wav16 = torchaudio.functional.resample(torch.from_numpy(clip), TARGET_SR, WHISPER_SR).numpy().astype(np.float32)
        segs, _ = model.transcribe(wav16, language="fr", vad_filter=False, beam_size=5)
        seg_list = [s for s in segs if s.text.strip()]
        if not seg_list:
            continue
        raw_tx = " ".join(s.text.strip() for s in seg_list)
        tx = prepare_reverb_text(raw_tx)
        lp = min(s.avg_logprob for s in seg_list)
        if len(tx) < 15:
            continue

        cid = f"clip_{len(rows):04d}"
        out = f"{AUDIO_OUT_DIR}/{cid}.wav"
        sf.write(out, clip, sr, subtype="PCM_16")
        rows[cid] = {"id": cid, "audio_path": f"audio/{cid}.wav", "text": tx, "raw_text": raw_tx, "language": "fr", "logprob": lp, "duration": len(clip) / sr}
        durs[cid] = len(clip) / sr

print(f"✅ {len(rows)} clips prêts ({sum(durs.values())/60:.1f} min)")
for r in sorted(rows.values(), key=lambda x: -x["duration"])[:5]:
    print(f"\n{r['id']} ({r['duration']:.1f}s) : {r['text'][:90]}")
    ipd.display(ipd.Audio(f"{WORK}/data/{r['audio_path']}"))

## 4. Filtrage locuteur (Resemblyzer) + Qualité + Export `dataset_reverb_narrateur.zip`

In [ ]:
import json, random, shutil
from resemblyzer import VoiceEncoder, preprocess_wav

kept = list(rows.values())
if NARRATOR_CLIP_ID and NARRATOR_CLIP_ID in rows:
    enc = VoiceEncoder(device=DEVICE)
    ref_emb = enc.embed_utterance(preprocess_wav(f"{WORK}/data/{rows[NARRATOR_CLIP_ID]['audio_path']}"))
    kept_sp = []
    for r in kept:
        emb = enc.embed_utterance(preprocess_wav(f"{WORK}/data/{r['audio_path']}"))
        if float(np.dot(ref_emb, emb)) >= SIM_THRESHOLD:
            kept_sp.append(r)
    print(f"Filtre locuteur : {len(kept_sp)} / {len(kept)} clips gardés")
    kept = kept_sp

kept = [r for r in kept if r["logprob"] >= MIN_LOGPROB]
print(f"Filtre qualité (logprob >= {MIN_LOGPROB}) : {len(kept)} clips gardés")
assert len(kept) >= 10, "Trop peu de clips après filtrage !"

random.seed(SPLIT_SEED)
random.shuffle(kept)
n_dev = max(1, int(len(kept) * DEV_RATIO))
dev, train = kept[:n_dev], kept[n_dev:]

for split_name, part in [("train", train), ("dev", dev)]:
    with open(f"{WORK}/data/{split_name}.jsonl", "w", encoding="utf-8") as f:
        for r in part:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Référence narrateur (5-10s idéale pour Reverb)
cands = [r for r in train if 4.0 <= r["duration"] <= 9.5]
ref = max(cands, key=lambda r: len(r["text"])) if cands else train[0]
shutil.copy2(f"{WORK}/data/{ref['audio_path']}", f"{WORK}/data/ref_narrateur.wav")
with open(f"{WORK}/data/ref_narrateur.txt", "w", encoding="utf-8") as f:
    f.write(ref["text"])

zip_out = f"{WORK}/dataset_reverb_narrateur"
shutil.make_archive(zip_out, "zip", WORK, "data")
drive_zip = f"{DRIVE_OUTPUT}/dataset_reverb_narrateur.zip"
shutil.copy2(f"{zip_out}.zip", drive_zip)
print(f"✅ Export terminé : {drive_zip} ({os.path.getsize(drive_zip)/1e6:.1f} Mo)")
print(f"   Train: {len(train)} clips | Dev: {len(dev)} clips")